In [0]:
import boto3

from pyspark.sql.functions import coalesce, concat_ws, substring, when, col, regexp_replace, trim
from pyspark.sql import DataFrame

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix
env = aws_utils.get_env_prefix()

In [0]:
# Takes the values of widgets
run_date = dbutils.widgets.get("run_date") 
# table_name = dbutils.widgets.get("table_name") 
table_name = 'DPortfolioToolThirdPartiesMonthly'
unity = dbutils.widgets.get("unity") 

# """
# Takes the s3 parameter value from ssm

# ssm_client = boto3.client("ssm", region_name="eu-central-1") # Initialize the SSM client

# response = ssm_client.get_parameter(
#     Name='/cfn/a360/matching/conf_s3_path',
#     WithDecryption=False
# )

# s3_path = response['Parameter']['Value']
# """

In [0]:
schema = (
"ExposureKey string,"
"PortfolioName string,"
"WorkableFlag boolean,"
# "WorkableFlagMonthly boolean,"
"NonWorkableReason string,"
"FrequencyType string,"
"DebtNumber string,"
# "NonWorkableReasonMonthly string,"
# "LastMonthMinFinalBucket string,"
"BusinessDate date"
)

In [0]:
partition_columns = ["BusinessDate"]
derivative_table_path = ""
 
# ================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
 
 
# ================== Delete From Delta Table ==================
 
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

In [0]:
calendar_df = spark.sql(f"""
SELECT CurrentMonthLastWorkingDate
FROM `{env}-cdh`.reference.ref_calendar
WHERE calendardate = add_months(DATE('{run_date}'), -1)
""")
last_working_day = calendar_df.collect()[0]["CurrentMonthLastWorkingDate"]
calendar_df2 = spark.sql(f"""
SELECT NextBusinessDay
FROM `{env}-cdh`.reference.ref_calendar
WHERE calendardate = '{last_working_day}'
""")


next_working_day = calendar_df2.collect()[0]["NextBusinessDay"]

# Ανάκτηση του "next of next"
next_of_next_df = spark.sql(f"""
SELECT NextBusinessDay AS NextOfNextBusinessDay
FROM `{env}-cdh`.reference.ref_calendar
WHERE calendardate = DATE('{next_working_day}')
""")

next_of_next_day = next_of_next_df.collect()[0]["NextOfNextBusinessDay"]
monthly_flag = 0
if str(run_date) == str(next_of_next_day):
    monthly_table = spark.sql(f"""
    SELECT DISTINCT
        m.ExposureKey,
        m.PortfolioName,
        m.LastWorkableFlagMonthly AS WorkableFlag,
        m.LastNonWorkableReasonMonthly AS NonWorkableReason,
        'Monthly' AS FrequencyType,
        cm.case_contract AS DebtNumber,
        m.BusinessDate
    FROM `{env}-cdh`.derivative.LastMonthEOM_WorkableFlagMonthly_NonWorkableReasonMonthly(
        CAST('{next_working_day}' AS DATE)
    ) m
    LEFT JOIN `{env}-cdh`.landed.qc_case_master cm
      ON m.ExposureKey = cm.old_case_code
     AND m.BusinessDate = cm.BusinessDate
    """)
    monthly_table.createOrReplaceTempView("monthly_table")
    monthly_flag = 1

In [0]:
if monthly_flag == 1:
    if unity == "True":
        monthly_table.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(derivative_table_path) # in order to write in delta_lake
    else:
        monthly_table.write \
            .format("delta") \
            .mode("append") \
            .save(derivative_table_path) # in order to write in delta_lake